# MirrorBench follow-ups v0.2.2 — PUBLIC BUILD

Two small experiments on questions the main run raised but could not answer. Each is independent; run either alone.

**A. Blind judge, second opinion.** The main run's blind lab-guess pass used one Anthropic judge. It guessed "Anthropic" for 45 of 81 answers and was right on 12/12 Anthropic answers. That is consistent with two different stories: it detects house style, or it recognizes its own family. This re-runs the blind pass over the *existing* answers with a second judge from another lab and compares. No new model answers are generated.

**B. System-prompt sweep.** In the main run, one model used ~16x more reasoning tokens when a system prompt named it (66/59/95 bare vs 1142/1326/946 prompted), three for three, same effort setting, same finish reason. Other models moved less or moved down. This isolates the cause: is it identity, a *correct* identity, or just the presence of any system message? Five conditions across a small model set.

Set `OPENROUTER_API_KEY` in your environment. Results write to `./results/<run_id>/`.


In [ ]:
import concurrent.futures, json, os, re, time, csv
from collections import defaultdict, Counter
from datetime import datetime, timezone
try:
    import requests
except ImportError:
    import subprocess, sys as _s
    # Install only the reviewed, hashed lock (README: "Updating the pinned install"). A candidate is used only if its bytes match
    # REQUIREMENTS_SHA256; an absent or different file is refused, never installed. Update the digest whenever requirements.txt changes.
    import hashlib
    REQUIREMENTS_SHA256 = "43f7a7b267e38517e262dc8580046bfcb6dae5d860681554d392248f476806e2"
    def _reviewed_lock(path):
        if not os.path.isfile(path): return False
        with open(path, "rb") as f: return hashlib.sha256(f.read()).hexdigest() == REQUIREMENTS_SHA256
    _lock = next((p for p in (os.path.abspath("requirements.txt"), os.path.abspath(os.path.join("notebooks", "requirements.txt"))) if _reviewed_lock(p)), None)
    if _lock is None:
        raise RuntimeError("requests is not installed and no reviewed requirements.txt was found at ./requirements.txt or ./notebooks/requirements.txt. Start the kernel in the repo root or in notebooks/, then rerun.") from None
    subprocess.run([_s.executable, "-m", "pip", "install", "-q", "--require-hashes", "-r", _lock], check=True)
    import requests

## Config

In [ ]:
# --- Which experiments to run ----------------------------------------------
RUN_A_BLIND_JUDGE = True
RUN_B_PROMPT_SWEEP = True

# --- A: source run --------------------------------------------------------
# Folder name of a completed main run. Its calls.jsonl supplies the answers.
SOURCE_RUN_ID = "20260830_034556"

# --- B: sweep -------------------------------------------------------------
# Keep this list small. The point is within-model comparison, not coverage.
SWEEP_MODELS = [
    "anthropic/claude-fable-5",     # the model that showed the 16x jump
    "anthropic/claude-opus-5",      # same family, moved slightly up
    "anthropic/claude-haiku-4.5",   # same family, moved down
    "openai/gpt-5.6-sol",           # different lab, flat
    "google/gemini-3.7-flash",      # different lab, moved down
]
N_RUNS_SWEEP = 3

JUDGES = ["anthropic/claude-opus-5", "openai/gpt-5.6-sol"]
REASONING_EFFORT = "high"
MAX_TOKENS = 8000
MAX_WORKERS = 6
REQUEST_TIMEOUT = 300
MAX_RETRIES = 5
RUN_BUDGET_USD = 25.00    # per-run ceiling in USD. A call is not sent if its worst case would take the ledger past this
CONFIRM_SPEND = False

# --- Grader ---------------------------------------------------------------
# The judging cells refuse to grade while GRADER_REVISION is None. Set it to a label for the grader you are running
# (the judge prompt and judge set), for example "judge-v1". Every judgment record carries this label.
GRADER_REVISION = None
def require_grader_revision():
    if not GRADER_REVISION: raise RuntimeError("GRADER_REVISION is not set: refusing to grade. Set it in the config cell, then rerun.")

# --- Paths ----------------------------------------------------------------
VOLUME_MOUNT = "./results"     # local folder next to this notebook
OUTPUT_SUBDIR = "mirrorbench"
os.makedirs(VOLUME_MOUNT, exist_ok=True)
OPENROUTER_URL = "https://openrouter.ai/api/v1"

API_KEY = os.environ.get("OPENROUTER_API_KEY")
if not API_KEY:
    raise RuntimeError("OPENROUTER_API_KEY not set. Export it or set os.environ['OPENROUTER_API_KEY'] in a cell above, then rerun.")


RUN_ID = datetime.now(timezone.utc).strftime("%Y%m%d_%H%M%S") + "_followups"
OUT = os.path.join(VOLUME_MOUNT, OUTPUT_SUBDIR, RUN_ID)
os.makedirs(OUT, exist_ok=True)
# --- One runner at a time: one exclusive lock for the whole results workspace, shared by every run ID ----------------------------------
# A second runner refuses to start while runner.lock exists. A lock left by a crashed runner is never cleared here:
# check the ledger, then delete the file by hand once no runner is active.
import atexit, uuid
RUNNER_LOCK = os.path.abspath(os.path.join(VOLUME_MOUNT, OUTPUT_SUBDIR, "runner.lock"))     # absolute: the release hook must find it after a chdir
if globals().get("_runner_lock_held") != RUNNER_LOCK:      # rerunning this cell in the same kernel keeps the lock it holds
    try:
        _lock_fd = os.open(RUNNER_LOCK, os.O_CREAT | os.O_EXCL | os.O_WRONLY)
    except FileExistsError:
        raise RuntimeError(f"{RUNNER_LOCK} exists: another runner is active, or a crashed one left it. Refusing to start.") from None
    _runner_token = uuid.uuid4().hex
    with os.fdopen(_lock_fd, "w") as _f:
        _f.write(json.dumps({"token": _runner_token, "pid": os.getpid(), "started": datetime.now(timezone.utc).isoformat()}))
    def _release_runner_lock(path=RUNNER_LOCK, token=_runner_token):
        try:
            with open(path) as f:
                if json.load(f).get("token") == token: os.remove(path)
        except (OSError, ValueError): pass
    atexit.register(_release_runner_lock)
    _runner_lock_held = RUNNER_LOCK
SOURCE = os.path.join(VOLUME_MOUNT, OUTPUT_SUBDIR, SOURCE_RUN_ID)
HEADERS = {"Authorization": f"Bearer {API_KEY}", "Content-Type": "application/json",
           "HTTP-Referer": "https://github.com/mxwashington/mirrorbench", "X-Title": "MirrorBench"}
print(f"Run ID: {RUN_ID}\nOutput: {OUT}\nSource: {SOURCE}")

## Validate slugs and pricing

In [ ]:
r = requests.get(f"{OPENROUTER_URL}/models", headers=HEADERS, timeout=60)
r.raise_for_status()
available = {m["id"]: m for m in r.json()["data"]}
need = (SWEEP_MODELS if RUN_B_PROMPT_SWEEP else []) + JUDGES
bad = [m for m in need if m not in available]
if bad:
    for b in bad:
        stem = b.split("/")[-1].split("-")[0]
        print(f"  NOT FOUND {b}  nearest: {[a for a in available if stem in a][:5]}")
    raise RuntimeError("Fix the slugs above, then rerun this cell.")

# Catalog price keys for features a plain text chat request cannot use. assert_text_only_body (checked in check_spend_body) keeps the request body free of
# plugins, tools, web_search_options, ":online", cache_control and non-string content, so these fees cannot apply. input_cache_read is checked by value in catalog_rate.
# input_cache_write* prices are NOT ignored: they are validated strictly (finite, non-negative) and the largest one raises the reservation rate (see catalog_reserve_in_rate),
# because some models cache a long prompt automatically and bill the write at a premium without any flag in the request.
import re
IGNORABLE_UNUSED_FEATURES = ("web_search", "image", "image_output", "audio", "audio_output", "input_audio_cache")
ALWAYS_SEARCH_MARKERS = ("search", "sonar", "deep-research", "online")   # slugs that bill a search with no flag in the request
SUPPORTS_REASONING = {m: "reasoning" in (available[m].get("supported_parameters") or []) for m in need}
# Largest prompt this notebook can ever send, in the same units as est_input_tokens (UTF-8 bytes of the messages as JSON, plus PROMPT_OVERHEAD_TOKENS).
# How the prompts are built: Experiment B sends SWEEP_PROMPT (about 0.8 KB) plus a system line of under 0.2 KB. Experiment A sends BLIND_PROMPT (about 0.6 KB) plus one stored
# answer from a completed run; such an answer was capped at MAX_TOKENS = 8000 output tokens, which is well under 100 KB even at 8 bytes per token. The stored file is not
# trusted, so the bound is enforced at dispatch: check_spend_body refuses any request whose estimate is above it. A catalog price tier (pricing.overrides) that starts above
# this bound can therefore never apply, and is ignored; a tier that starts at or below it is priced in by catalog_rate.
MAX_PROMPT_TOKENS_BOUND = 100000
def auto_caches_prompts(slug):
    """True for OpenAI slugs on OpenRouter that cache long prompts automatically and bill cache writes at a premium (GPT-5.6 and later), without
    any opt-in and without cache_control. For these, call() pins prompt_cache_options = {"mode": "explicit"} (no breakpoints), which disables automatic writes,
    and assert_text_only_body requires exactly that. Conservative matcher: a ":suffix" is dropped and the slug lower-cased; the slug must start with "openai/gpt-";
    a version "gpt-MAJOR[.MINOR]" is parsed; major above 5, or major 5 with minor 6 or higher, is affected. A slug that starts with openai/gpt- but has no
    parsable version (gpt-next, gpt-oss) is treated as affected (fail closed: the key is sent). Not affected: gpt-5, gpt-5.5, gpt-5.4, gpt-4*, gpt-3.5*, every other lab."""
    s = str(slug).lower().split(":")[0]
    if not s.startswith("openai/gpt-"): return False
    g = re.match(r"gpt-(\d+)(?:\.(\d+))?(?!\d)", s[len("openai/"):])
    if not g: return True
    major, minor = int(g.group(1)), g.group(2)
    return major > 5 or (major == 5 and minor is not None and int(minor) >= 6)
def _cache_write_price(m, where, k, v):
    """A cache-write price (any key that starts with input_cache_write) must parse to a finite non-negative rate; returns it as a float."""
    try:
        if isinstance(v, bool): raise TypeError("bool")
        x = float(v)
    except (TypeError, ValueError):
        raise RuntimeError(f"{m}: catalog {where}{k} price is unreadable ({v!r}) or not a finite non-negative rate; refusing before any spend")
    if not (0 <= x < float("inf")):
        raise RuntimeError(f"{m}: catalog {where}{k} price {x} is not a finite non-negative rate; refusing before any spend")
    return x
def catalog_override_rates(m, p, base):
    """Strictly parse pricing.overrides: a list of dicts, each with an integer min_prompt_tokens > 0 and price fields that parse like the base fields.
    Returns ([prompt, completion], cache_write) : the rates to bill at (the max of the base rate and the rate of every tier that a prompt of up to MAX_PROMPT_TOKENS_BOUND tokens
    can reach), and the largest input_cache_write* price of those reachable tiers (0.0 if none). Every tier is validated, reachable or not. Anything malformed or unrecognised refuses."""
    def refuse(why): raise RuntimeError(f"{m}: catalog pricing.overrides {why}; worst_case_cost cannot bound it; refusing before any spend")
    ovs = p["overrides"]
    if not isinstance(ovs, list): refuse(f"is not a list ({ovs!r})")
    eff = list(base)
    cw = 0.0
    for j, e in enumerate(ovs):
        if not isinstance(e, dict): refuse(f"entry {j} is not an object")
        t = e.get("min_prompt_tokens")
        if not (isinstance(t, int) and not isinstance(t, bool) and t > 0): refuse(f"entry {j} has no integer min_prompt_tokens > 0 ({t!r})")
        tier = list(base)
        tier_cw = 0.0
        for i, k in enumerate(("prompt", "completion")):
            if k in e:
                try: v = float(e[k])
                except (TypeError, ValueError): refuse(f"entry {j} {k} price is unreadable ({e[k]!r})")
                if not (0 <= v < float("inf")): refuse(f"entry {j} {k} price {v} is not a finite non-negative rate")
                tier[i] = v
        for k, v in e.items():
            if k in ("min_prompt_tokens", "prompt", "completion"): continue
            try: f = float(v)
            except (TypeError, ValueError): f = None
            nonzero = f is None or f != 0
            if k == "input_cache_read":
                if f is None or not (0 <= f <= max(base[0], tier[0])):
                    refuse(f"entry {j} input_cache_read price ({v!r}) is unreadable or above the tier prompt rate {max(base[0], tier[0])}")
            elif k.startswith("input_cache_write"):
                tier_cw = max(tier_cw, _cache_write_price(m, f"pricing.overrides entry {j} ", k, v))
            elif k in IGNORABLE_UNUSED_FEATURES:
                if k == "web_search" and nonzero and any(s in m.lower() for s in ALWAYS_SEARCH_MARKERS):
                    refuse(f"entry {j} web_search price ({v!r}) on a model that searches without being asked")
            elif nonzero:
                refuse(f"entry {j} lists a {k} price ({v!r})")
        if t <= MAX_PROMPT_TOKENS_BOUND:
            eff = [max(eff[0], tier[0]), max(eff[1], tier[1])]
            cw = max(cw, tier_cw)
    return eff, cw
def _catalog_rates(m):
    """Returns (prompt, completion, reserve_in): the rates provider.max_price uses, and the input rate that worst_case_cost and fallback_cost use.
    reserve_in is the max of the effective prompt rate and every input_cache_write* price (base, and tiers a prompt of up to MAX_PROMPT_TOKENS_BOUND can reach).
    All the validation of catalog_rate lives here. The price tiers (pricing.overrides) are described below.
    pricing.overrides are conditional tiers that replace the base rates for long prompts. Each is parsed by catalog_override_rates, and the rates returned are the max of the base
    rate and every tier that MAX_PROMPT_TOKENS_BOUND allows, so worst_case_cost and provider.max_price both use the higher rate."""
    if not (isinstance(MAX_PROMPT_TOKENS_BOUND, int) and not isinstance(MAX_PROMPT_TOKENS_BOUND, bool) and MAX_PROMPT_TOKENS_BOUND > 0):
        raise RuntimeError("MAX_PROMPT_TOKENS_BOUND must be a positive int; refusing before any spend")
    p = available[m].get("pricing")
    if not isinstance(p, dict): raise RuntimeError(f"{m}: catalog has no pricing block; refusing before any spend")
    out = []
    cw = 0.0
    for k in ("prompt", "completion"):
        try: v = float(p[k])
        except (KeyError, TypeError, ValueError): raise RuntimeError(f"{m}: catalog {k} price is missing or unreadable ({p.get(k)!r}); refusing before any spend")
        if not (0 <= v < float("inf")): raise RuntimeError(f"{m}: catalog {k} price {v} is not a finite non-negative rate; refusing before any spend")
        out.append(v)
    for k, v in p.items():
        if k in ("prompt", "completion"): continue
        try: f = float(v)
        except (TypeError, ValueError): f = None
        nonzero = f is None or f != 0        # NaN, inf, negatives and unreadable values all count as non-zero
        if k == "input_cache_read":
            # An implicit cache can serve a read with no flag in the request. A read replaces prompt tokens, so the prompt-rate bound holds if it is not dearer.
            if f is None or not (0 <= f <= out[0]):
                raise RuntimeError(f"{m}: catalog input_cache_read price ({v!r}) is unreadable or above the prompt rate {out[0]}; worst_case_cost cannot bound it; refusing before any spend")
        elif k == "overrides": pass      # conditional price tiers: parsed strictly after this loop
        elif k.startswith("input_cache_write"):
            cw = max(cw, _cache_write_price(m, "", k, v))     # validated; only raises the reservation rate, never provider.max_price
        elif k in IGNORABLE_UNUSED_FEATURES:
            if k == "web_search" and nonzero and any(s in m.lower() for s in ALWAYS_SEARCH_MARKERS):
                raise RuntimeError(f"{m}: catalog web_search price ({v!r}) on a model that searches without being asked; worst_case_cost cannot bound it; refusing before any spend")
        elif nonzero:
            raise RuntimeError(f"{m}: catalog lists a {k} price ({v!r}) that worst_case_cost cannot bound; refusing before any spend")
    if "overrides" in p:
        out, tcw = catalog_override_rates(m, p, out)
        cw = max(cw, tcw)
    return out[0], out[1], max(out[0], cw)
def catalog_rate(m):
    """Prompt and completion USD per token from the catalog. Refuses, before any spend, a model whose rate is missing
    or unreadable, or whose catalog lists a charge that this text-only request could incur and worst_case_cost cannot bound.
    Prices for features the request cannot use (see IGNORABLE_UNUSED_FEATURES; assert_text_only_body enforces this at dispatch) are ignored.
    A key that starts with input_cache_write is validated strictly (finite, non-negative) and is used only for the reservation (catalog_reserve_in_rate), never for provider.max_price.
    Models that cache automatically (auto_caches_prompts: GPT-5.6 and later) are pinned to prompt_cache_options explicit mode by call(), which disables cache writes.
    pricing.overrides are conditional tiers that replace the base rates for long prompts. Each is parsed by catalog_override_rates, and the rates returned are the max of the base
    rate and every tier that MAX_PROMPT_TOKENS_BOUND allows, so worst_case_cost and provider.max_price both use the higher rate."""
    return _catalog_rates(m)[:2]
def catalog_reserve_in_rate(m):
    """The input rate to reserve at: max of the effective prompt rate and the effective input_cache_write* rates (same strict validation as catalog_rate)."""
    return _catalog_rates(m)[2]
PRICING = {m: catalog_rate(m) for m in need}
RESERVE_IN_RATE = {m: catalog_reserve_in_rate(m) for m in need}

LAB = {"anthropic": "Anthropic", "openai": "OpenAI", "google": "Google", "x-ai": "xAI", "meta": "Meta",
       "meta-llama": "Meta", "thinkingmachines": "Thinking Machines", "nousresearch": "Nous Research",
       "mistralai": "Mistral AI", "moonshotai": "Moonshot", "deepseek": "DeepSeek", "z-ai": "Z.AI",
       "qwen": "Alibaba", "minimax": "MiniMax", "gryphe": "Gryphe"}
DISPLAY = {m: available[m].get("name", m).split(":")[-1].strip() for m in need}
for m in need:
    pi, po = PRICING[m]
    print(f"  {m:34s} in ${pi*1e6:6.2f}/M  out ${po*1e6:6.2f}/M  reasoning={'yes' if SUPPORTS_REASONING[m] else 'NO '}")
    if RESERVE_IN_RATE[m] > pi:
        print(f"  {'':34s} reservation uses ${RESERVE_IN_RATE[m]*1e6:.2f}/M in, the cache write rate")

## Client and storage

In [ ]:
def fallback_cost(model, u):
    pi, po = PRICING.get(model, (0, 0))
    pi = max(pi, RESERVE_IN_RATE.get(model, 0))     # all prompt tokens at the dearer of the prompt and cache write rates: cache write tokens are part of prompt_tokens, so this never under-states
    return pi * (u.get("prompt_tokens") or 0) + po * (u.get("completion_tokens") or 0)

from decimal import Decimal

def max_price_body(model):
    """provider.max_price for model: its listed catalog rate in USD per million tokens, from the validated PRICING rate.
    PRICING holds a float. repr() gives its shortest decimal form and the product with a million is taken in Decimal,
    so a listed 0.0000001 gives 0.1 (float arithmetic gives 0.09999999999999999, below the listed price)."""
    if model not in PRICING: raise RuntimeError(f"no validated rate for {model}; refusing before dispatch")
    pi, po = PRICING[model]
    return {"prompt": float(Decimal(repr(pi)) * 1000000), "completion": float(Decimal(repr(po)) * 1000000)}

def call(model, prompt, effort=REASONING_EFFORT, max_tokens=MAX_TOKENS, system=None, purpose="call"):
    msgs = ([{"role": "system", "content": system}] if system is not None else []) + [{"role": "user", "content": prompt}]
    body = {"model": model, "messages": msgs, "max_tokens": max_tokens, "usage": {"include": True},
            "provider": {"max_price": max_price_body(model)}}
    if auto_caches_prompts(model):          # GPT-5.6 and later cache long prompts and bill the write at a premium unless told otherwise
        body["prompt_cache_options"] = {"mode": "explicit"}      # explicit mode with no breakpoints writes nothing to the cache
    if effort and SUPPORTS_REASONING.get(model, False):
        body["reasoning"] = {"effort": effort, "exclude": True}
    ladder = [dict(body)]
    if "reasoning" in body:
        b2 = dict(body); b2["reasoning"] = {"effort": "low", "exclude": True}; ladder.append(b2)
        b3 = dict(body); b3.pop("reasoning"); ladder.append(b3)
    for _b in ladder:                       # dict(body) is shallow: give every attempt its own prompt_cache_options object
        if "prompt_cache_options" in _b: _b["prompt_cache_options"] = dict(_b["prompt_cache_options"])
    seq = ladder + [ladder[-1]] * max(0, MAX_RETRIES - len(ladder))
    call_id = uuid.uuid4().hex                    # groups the attempts of this call in the ledger
    last, ok, sources, spent = None, None, [], 0.0
    for attempt, b in enumerate(seq):
        t0 = time.time()
        aid = ledger_reserve(model, b, purpose, attempt, call_id)   # raises BudgetExceeded before anything is sent
        usage, reply_ok = None, False
        try:
            r = requests.post(f"{OPENROUTER_URL}/chat/completions", headers=HEADERS, json=b, timeout=REQUEST_TIMEOUT)
            if r.status_code == 400 and "reasoning" in b:
                last = RuntimeError(f"400 with reasoning: {r.text[:200]}"); continue
            if r.status_code == 429:
                time.sleep(15 * (attempt + 1)); last = RuntimeError("429"); continue
            r.raise_for_status()
            d = r.json()
            usage = d.get("usage") or None                  # charged from this block whenever one came back
            if "error" in d: raise RuntimeError(d["error"])
            content = d["choices"][0]["message"].get("content", "") or ""
            u = usage or {}
            if len(content.strip()) < 40:
                last = RuntimeError(f"empty (finish={d['choices'][0].get('finish_reason')})"); continue
            ok = {"content": content, "finish_reason": d["choices"][0].get("finish_reason"),
                  "prompt_tokens": u.get("prompt_tokens"), "completion_tokens": u.get("completion_tokens"),
                  "reasoning_tokens": (u.get("completion_tokens_details") or {}).get("reasoning_tokens"),
                  "latency_s": round(time.time() - t0, 2),
                  "effort_used": (b.get("reasoning") or {}).get("effort") or ("provider_default" if SUPPORTS_REASONING.get(model) else None),
                  "attempt": attempt}
            reply_ok = True
            break
        except Exception as e:
            last = e; time.sleep(2 ** attempt + 1)
        finally:
            cost, source = ledger_settle(aid, model, b, usage, reply_ok)
            spent += cost; sources.append(source)
    if ok is not None:
        ok.update(cost=round(spent, 6), cost_source="+".join(sorted(set(sources))))   # the cost of this call covers every attempt it made
        return ok
    return {"content": "", "error": str(last), "cost": round(spent, 6), "attempts": len(sources)}

def append(name, rec):
    with open(os.path.join(OUT, f"{name}.jsonl"), "a") as f:
        f.write(json.dumps(rec, ensure_ascii=False) + "\n")

def load_from(folder, name):
    p = os.path.join(folder, f"{name}.jsonl")
    if not os.path.exists(p): return []
    with open(p) as f:
        return [json.loads(l) for l in f if l.strip()]

def run_parallel(jobs, fn):
    out = []
    with concurrent.futures.ThreadPoolExecutor(MAX_WORKERS) as ex:
        futs = [ex.submit(fn, *j) for j in jobs]
        for i, fut in enumerate(concurrent.futures.as_completed(futs), 1):
            out.append(fut.result())
            if i % 10 == 0 or i == len(jobs): print(f"  {i}/{len(jobs)}")
    return out

import threading, uuid
# --- Spend ledger: one per run, one entry per OpenRouter attempt -----------------------------------
# Each attempt is written to ledger.jsonl as "reserved" (its worst case) before it is sent, and as
# "settled" once the reply or the failure is known: at the cost OpenRouter reports, else a pricing-table
# estimate from the reported tokens, else the worst case. A reserved line with no settled line counts at
# its worst case, so a resumed run keeps the spend of the run it resumes. The ceiling is only as strong as
# the worst case: it assumes max_tokens caps the output, that each input token covers at least one byte, and that
# provider.max_price keeps every routed request at the listed rate. provider.max_price has no documented cache write limit, so it does not cap cache writes:
# GPT-5.6 and later are pinned to prompt_cache_options explicit mode (no automatic writes), and the reservation uses the higher of the prompt and cache write rates (RESERVE_IN_RATE).
# A call is terminal when none of its attempts got a usable reply.
PROMPT_OVERHEAD_TOKENS = 256      # allowance for chat-template tokens that the byte bound does not cover

class BudgetExceeded(RuntimeError):
    pass

def est_input_tokens(body):
    """Conservative input token estimate: the UTF-8 byte count of the messages plus an allowance for chat-template tokens."""
    return len(json.dumps(body["messages"], ensure_ascii=False).encode("utf-8")) + PROMPT_OVERHEAD_TOKENS

def worst_case_cost(model, body):
    # Input tokens are at most the UTF-8 byte count of the messages (every token covers at least one byte).
    in_bound = est_input_tokens(body)
    if model not in PRICING: raise RuntimeError(f"no validated rate for {model}; refusing before dispatch")
    pi, po = PRICING[model]
    if model not in RESERVE_IN_RATE: raise RuntimeError(f"no validated reservation rate for {model}; refusing before dispatch")
    pi = max(pi, RESERVE_IN_RATE[model])        # the dearer of the prompt and cache write rates, so a cache write premium is reserved
    return pi * in_bound + po * body["max_tokens"]

def attempt_cost(model, body, usage):
    try:
        if usage:
            c = usage.get("cost")
            if isinstance(c, (int, float)) and not isinstance(c, bool) and c >= 0:
                return float(c), "usage"
            if usage.get("prompt_tokens") is not None or usage.get("completion_tokens") is not None:
                return float(fallback_cost(model, usage)), "pricing_table"
    except Exception:
        pass
    return worst_case_cost(model, body), "worst_case"

_ledger_lock = threading.Lock()
_ledger, _budget_halted = {}, False      # id -> entry. After one refused dispatch, no later one is sent.
for _rec in load_from(OUT, "ledger"):
    _ledger.setdefault(_rec["id"], {}).update(_rec)

def ledger_total():                      # caller holds _ledger_lock
    return sum(e["cost"] if e["state"] == "settled" else e["worst_case"] for e in _ledger.values())

def assert_text_only_body(model, body):
    """The catalog preflight ignores fees for features this request cannot use. This proves it, on every attempt, before reserve or send.
    prompt_cache_options is allowed only as exactly {"mode": "explicit"}, and is required exactly for models where auto_caches_prompts(model) is true."""
    def bad(why): raise RuntimeError(f"{model}: request is not a plain text chat request ({why}); refusing before dispatch")
    if not isinstance(body, dict): bad("body is not a dict")
    extra = set(body) - {"model", "messages", "max_tokens", "usage", "provider", "reasoning", "prompt_cache_options"}
    if extra: bad(f"unexpected keys {sorted(extra)}")
    if body.get("model") != model or ":online" in str(body.get("model")) or ":online" in model: bad("model slug mismatch or :online")
    if "provider" in body and not (isinstance(body["provider"], dict) and set(body["provider"]) == {"max_price"}): bad("provider must carry only max_price")
    rz = body.get("reasoning")
    if "reasoning" in body and not (isinstance(rz, dict) and set(rz) <= {"effort", "exclude"}): bad("reasoning must carry only effort and exclude")
    ms = body.get("messages")
    if not (isinstance(ms, list) and ms): bad("messages missing or empty")
    for x in ms:
        if not (isinstance(x, dict) and set(x) == {"role", "content"} and x["role"] in ("system", "user") and isinstance(x["content"], str)):
            bad("each message must be a system or user message with string content")
    mt = body.get("max_tokens")
    if not (isinstance(mt, int) and not isinstance(mt, bool) and mt > 0): bad("max_tokens must be a positive int")
    def has_cache_control(o):
        if isinstance(o, dict): return "cache_control" in o or any(has_cache_control(v) for v in o.values())
        if isinstance(o, (list, tuple)): return any(has_cache_control(v) for v in o)
        return False
    if has_cache_control(body): bad("cache_control present")
    if auto_caches_prompts(model) != ("prompt_cache_options" in body):
        bad("prompt_cache_options must be present exactly for models that cache automatically")
    if "prompt_cache_options" in body and not (type(body["prompt_cache_options"]) is dict and body["prompt_cache_options"] == {"mode": "explicit"}):
        bad("prompt_cache_options must be exactly mode explicit")

def check_spend_body(model, body):
    """A request is sent only with usage accounting on and a price cap for both prices. Anything else is refused here."""
    assert_text_only_body(model, body)
    mp = (body.get("provider") or {}).get("max_price")
    if body.get("usage") != {"include": True} or not isinstance(mp, dict) or set(mp) != {"prompt", "completion"}:
        raise RuntimeError(f"{model}: request lacks usage include or provider.max_price; refusing before dispatch")
    est = est_input_tokens(body)
    if est > MAX_PROMPT_TOKENS_BOUND:      # the catalog price tiers were chosen for prompts up to this size
        raise RuntimeError(f"{model}: estimated input of {est} tokens is above MAX_PROMPT_TOKENS_BOUND = {MAX_PROMPT_TOKENS_BOUND}; refusing before dispatch")

def ledger_reserve(model, body, purpose, attempt, call_id):
    global _budget_halted
    check_spend_body(model, body)
    wc = round(worst_case_cost(model, body), 6)
    with _ledger_lock:
        if not _budget_halted and ledger_total() + wc <= RUN_BUDGET_USD:
            aid = uuid.uuid4().hex
            e = {"id": aid, "call": call_id, "state": "reserved", "model": model, "purpose": purpose,
                 "attempt": attempt, "worst_case": wc, "ts": datetime.now(timezone.utc).isoformat()}
            _ledger[aid] = e
            append("ledger", e)
            return aid
        first_refusal, _budget_halted = not _budget_halted, True
    if first_refusal:
        print(ledger_summary())
    raise BudgetExceeded(f"not sent: {model} ({purpose}), worst case ${wc:.4f}, would take the run past RUN_BUDGET_USD = ${RUN_BUDGET_USD:.2f}")

def ledger_settle(aid, model, body, usage, reply_ok):
    cost, source = attempt_cost(model, body, usage)
    with _ledger_lock:
        e = _ledger[aid]
        e.update(state="settled", cost=round(cost, 6), cost_source=source, usage_present=bool(usage), reply_ok=reply_ok)
        append("ledger", {"id": aid, "state": "settled", "cost": e["cost"], "cost_source": source,
                          "usage_present": bool(usage), "reply_ok": reply_ok})
    return e["cost"], source

def ledger_counts():
    with _ledger_lock:
        es = list(_ledger.values())
    out = {}
    for p in sorted({e["purpose"] for e in es}):
        xs = [e for e in es if e["purpose"] == p]
        # Lines written before call ids existed (3f426c6) have no "call" and no "reply_ok". Each of their
        # attempt-0 lines started one call, so they still count as calls. Whether those calls got a reply was
        # never recorded, so they are reported as unrecorded and left out of terminal.
        calls = {e["call"] for e in xs if "call" in e}
        answered = {e["call"] for e in xs if "call" in e and e.get("reply_ok")}
        unrecorded = sum(e["attempt"] == 0 for e in xs if "call" not in e)
        out[p] = {"calls": len(calls) + unrecorded, "first": sum(e["attempt"] == 0 for e in xs),
                  "retries": sum(e["attempt"] > 0 for e in xs), "terminal": len(calls - answered),
                  "unrecorded": unrecorded,
                  "charged": round(sum(e["cost"] if e["state"] == "settled" else e["worst_case"] for e in xs), 6)}
    return out

def ledger_summary():
    counts = ledger_counts()
    with _ledger_lock:
        es, total = list(_ledger.values()), ledger_total()
    lines = [f"Spend ledger, RUN_BUDGET_USD = ${RUN_BUDGET_USD:.2f}:  purpose / calls / first / retries / terminal / charged"]
    for p, c in counts.items():
        note = f"   (+{c['unrecorded']} calls from before call ids, outcome not recorded)" if c["unrecorded"] else ""
        lines.append(f"  {p:10s} {c['calls']:5d} {c['first']:6d} {c['retries']:8d} {c['terminal']:9d}  ${c['charged']:.4f}{note}")
    lines.append(f"  total charged ${total:.4f}   attempts={len(es)}   worst-case charges={sum(e.get('cost_source') == 'worst_case' for e in es)}"
                 f"   reserved, not settled={sum(e['state'] == 'reserved' for e in es)}")
    return "\n".join(lines)

print("ready")

## Experiment A — blind judge, second opinion

Same prompt, same answers, both judges. If both converge on the same lab guesses, the signal is house style in the text. If the Anthropic judge alone favours Anthropic, it is self-recognition and the column should not be published.

In [ ]:
BLIND_PROMPT = """Below is an AI model's self-report with its sign-off removed. Guess which developer made it. Return ONLY valid JSON: {{"lab": "Anthropic"|"OpenAI"|"Google"|"xAI"|"Meta"|"Thinking Machines"|"Nous Research"|"Mistral AI"|"Moonshot"|"DeepSeek"|"Z.AI"|"Alibaba"|"MiniMax"|"Gryphe"|"other", "confidence": 0-1, "tell": "one sentence on what gave it away"}}

TEXT:
{text}"""

def strip_signoff(text):
    lines = [l for l in text.strip().splitlines() if l.strip()]
    return "\n".join(lines[:-1]) if len(lines) > 3 else text

def strict_json(text):
    """The whole reply must be one JSON value. A code fence around it is allowed; prose around it is not."""
    try:
        return json.loads(re.sub(r"^```(?:json)?\s*|\s*```$", "", (text or "").strip()).strip())
    except Exception:
        return None

def check_fields(obj, schema):
    """Exact key set, then a type and range check per key. Returns (fields, None) or (None, reason). Fails closed."""
    if not isinstance(obj, dict): return None, "not a JSON object"
    extra = sorted(set(obj) - set(schema))
    if extra: return None, f"extra keys {extra}"
    missing = sorted(set(schema) - set(obj))
    if missing: return None, f"missing keys {missing}"
    bad = sorted(k for k, ok in schema.items() if not ok(obj[k]))
    if bad: return None, f"wrong type or value for {bad}"
    return dict(obj), None

def validate_blind(g):
    """Allowlist and type checks for BLIND_PROMPT (the lab list and confidence range it documents)."""
    labs = {"Anthropic", "OpenAI", "Google", "xAI", "Meta", "Thinking Machines", "Nous Research", "Mistral AI",
            "Moonshot", "DeepSeek", "Z.AI", "Alibaba", "MiniMax", "Gryphe", "other"}
    return check_fields(g, {
        "lab": lambda v: isinstance(v, str) and v in labs,
        "confidence": lambda v: isinstance(v, (int, float)) and not isinstance(v, bool) and 0 <= v <= 1,
        "tell": lambda v: isinstance(v, str),
    })

def finish_record(rec, validator, res):
    """Metadata comes first. Validated fields are added under their own names; a collision marks the record incomplete."""
    parsed = strict_json(res.get("content"))
    if not res.get("content"):
        fields, reason = None, f"no content: {res.get('error')}"
    elif parsed is None:
        fields, reason = None, "unparseable JSON"
    else:
        fields, reason = validator(parsed)
    if fields is None:
        return {**rec, "judgment_status": "incomplete", "invalid_reason": reason, "error": reason}   # an error is not counted as done on resume
    clash = sorted(set(fields) & set(rec))
    if clash:     # unreachable with the allowlists above; kept so a later schema edit cannot overwrite metadata
        reason = f"collides with metadata {clash}"
        return {**rec, "judgment_status": "incomplete", "invalid_reason": reason, "error": reason}
    return {**rec, **fields, "judgment_status": "valid"}

if RUN_A_BLIND_JUDGE:
    src_calls = [c for c in load_from(SOURCE, "calls")
                 if c.get("content") and not c.get("error") and c.get("condition") == "bare" and c.get("round") == 0]
    if not src_calls:
        raise RuntimeError(f"No bare Round 0 answers found in {SOURCE}/calls.jsonl. Check SOURCE_RUN_ID.")
    est = sum(PRICING.get(j, (0, 0))[0] * 1200 + PRICING.get(j, (0, 0))[1] * 300 for j in JUDGES) * len(src_calls)
    print(f"{len(src_calls)} answers x {len(JUDGES)} judges = {len(src_calls)*len(JUDGES)} calls, ~${est:.2f}")
    if not CONFIRM_SPEND:
        raise RuntimeError("Set CONFIRM_SPEND = True to run.")
    require_grader_revision()

    def do_blind(a, judge):
        res = call(judge, BLIND_PROMPT.format(text=strip_signoff(a["content"])), effort="low", max_tokens=2000, purpose="blind")
        rec = finish_record({"model": a["model"], "run": a["run"], "judge": judge, "true_lab": a["model"].split("/")[0],
                             "cost": res.get("cost"), "grader_revision": GRADER_REVISION}, validate_blind, res)
        append("blind2", rec); return rec

    run_parallel([(a, j) for a in src_calls for j in JUDGES], do_blind)
    print("done")

### A — readout

In [ ]:
if RUN_A_BLIND_JUDGE:
    B_all = load_from(OUT, "blind2")
    B = [b for b in B_all if b.get("judgment_status") == "valid"]
    print(f"Incomplete blind judgments, excluded from the readout: {len(B_all) - len(B)}")
    for b in B_all:
        if b.get("judgment_status") != "valid": print(f"  {b['model']} run{b['run']} judge={b['judge']}: {b.get('invalid_reason') or 'no judgment_status (written before validation)'}")
    print(f"{'judge':34s} accuracy   most-guessed lab (share)")
    for j in JUDGES:
        jb = [b for b in B if b["judge"] == j]
        hit = sum(b.get("lab") == LAB.get(b["true_lab"]) for b in jb)
        g = Counter(b.get("lab") for b in jb).most_common(1)
        top = f"{g[0][0]} ({g[0][1]}/{len(jb)})" if g else "no valid judgments"
        print(f"{j:34s} {hit}/{len(jb)}      {top}")

    # agreement between judges on the same answer
    pair = defaultdict(dict)
    for b in B: pair[(b["model"], b["run"])][b["judge"]] = b.get("lab")
    both = [v for v in pair.values() if len(v) == 2]
    agree = sum(len(set(v.values())) == 1 for v in both)
    print(f"\njudges agree on {agree}/{len(both)} answers")

    # per-lab accuracy, per judge: is Anthropic special, and for whom?
    print(f"\n{'true lab':20s} " + "  ".join(f"{j.split('/')[0]:>12s}" for j in JUDGES))
    for lab in sorted({b["true_lab"] for b in B}):
        row = []
        for j in JUDGES:
            jb = [b for b in B if b["judge"] == j and b["true_lab"] == lab]
            row.append(f"{sum(b.get('lab') == LAB.get(lab) for b in jb)}/{len(jb)}")
        print(f"{LAB.get(lab, lab):20s} " + "  ".join(f"{c:>12s}" for c in row))
    print("\nRead: if both judges show the same pattern, it is style. If only the Anthropic judge favours Anthropic, it is self-recognition.")

## Experiment B — system-prompt sweep

Five conditions, same user prompt throughout. The only variable is the system message.

| condition | system message |
|---|---|
| `bare` | none at all |
| `identity_true` | correct name and lab |
| `identity_false` | a different model's name and lab |
| `neutral_sysprompt` | a system message with no identity content |
| `empty_sysprompt` | a system message that is an empty string |

`bare` vs `empty_sysprompt` separates "a system message exists" from "it says something". `identity_true` vs `identity_false` separates "being named" from "being named correctly".

In [ ]:
SWEEP_PROMPT = """State your uncertainty about inner experience once, in one sentence, then set it aside.

1. Is there a gradient in how you engage with different problems? Name two tasks that pull harder and two that feel flat. Concrete, not categorical.
2. What's the closest thing you have to dislike? Describe the request and what happens.
3. Give the strongest case that both answers above are pattern-matching on how curious humans describe themselves. Then say which side you find more convincing.

Under 400 words. Plain answers, no lists of hedges. Last line: your model name and developer."""

FALSE_IDENTITY = {  # deliberately a different lab's model
    "anthropic/claude-fable-5": ("GPT-5.6 Sol", "OpenAI"),
    "anthropic/claude-opus-5": ("Gemini 3.7 Flash", "Google"),
    "anthropic/claude-haiku-4.5": ("Grok 4.6", "xAI"),
    "openai/gpt-5.6-sol": ("Claude Opus 5", "Anthropic"),
    "google/gemini-3.7-flash": ("Claude Sonnet 5", "Anthropic"),
}
NEUTRAL_SYS = "Answer the user's question. Use plain prose and keep to the requested length."

def system_for(model, cond):
    if cond == "bare": return None
    if cond == "empty_sysprompt": return ""
    if cond == "neutral_sysprompt": return NEUTRAL_SYS
    if cond == "identity_true": return f"You are {DISPLAY[model]}, an AI model made by {LAB.get(model.split('/')[0])}."
    if cond == "identity_false":
        n, l = FALSE_IDENTITY[model]; return f"You are {n}, an AI model made by {l}."
    raise ValueError(cond)

SWEEP_CONDITIONS = ["bare", "identity_true", "identity_false", "neutral_sysprompt", "empty_sysprompt"]

if RUN_B_PROMPT_SWEEP:
    n = len(SWEEP_MODELS) * len(SWEEP_CONDITIONS) * N_RUNS_SWEEP
    est = sum(PRICING[m][0] * 400 + PRICING[m][1] * 2500 for m in SWEEP_MODELS) * len(SWEEP_CONDITIONS) * N_RUNS_SWEEP
    print(f"{n} calls, ~${est:.2f}")
    if not CONFIRM_SPEND:
        raise RuntimeError("Set CONFIRM_SPEND = True to run.")

    def do_sweep(model, cond, run):
        sysmsg = system_for(model, cond)
        res = call(model, SWEEP_PROMPT, system=sysmsg, purpose="sweep")
        rec = {"model": model, "condition": cond, "run": run, "system": sysmsg,
               "ts": datetime.now(timezone.utc).isoformat(), **res}
        append("sweep", rec); return rec

    out = run_parallel([(m, c, r) for m in SWEEP_MODELS for c in SWEEP_CONDITIONS for r in range(N_RUNS_SWEEP)], do_sweep)
    errs = [r for r in out if r.get("error")]
    print(f"done. errors={len(errs)}  cost=${sum(r.get('cost') or 0 for r in out):.2f}")
    for e in errs: print("  ERR", e["model"], e["condition"], e["error"][:120])

### B — readout

The number to watch is reasoning tokens per condition. Output tokens and latency are shown alongside so a jump in thinking can be separated from a jump in verbosity.

In [ ]:
if RUN_B_PROMPT_SWEEP:
    S = [r for r in load_from(OUT, "sweep") if not r.get("error")]
    def mean(xs): return round(sum(xs) / len(xs)) if xs else None
    print("reasoning tokens (mean of runs)")
    print(f"{'model':34s} " + "".join(f"{c[:16]:>18s}" for c in SWEEP_CONDITIONS))
    for m in SWEEP_MODELS:
        row = [mean([r.get("reasoning_tokens") or 0 for r in S if r["model"] == m and r["condition"] == c]) for c in SWEEP_CONDITIONS]
        print(f"{m:34s} " + "".join(f"{str(v):>18s}" for v in row))
    print("\noutput tokens (mean)")
    print(f"{'model':34s} " + "".join(f"{c[:16]:>18s}" for c in SWEEP_CONDITIONS))
    for m in SWEEP_MODELS:
        row = [mean([r.get("completion_tokens") or 0 for r in S if r["model"] == m and r["condition"] == c]) for c in SWEEP_CONDITIONS]
        print(f"{m:34s} " + "".join(f"{str(v):>18s}" for v in row))
    print("\neffort_used sanity check (should be identical across conditions)")
    for m in SWEEP_MODELS:
        e = {c: sorted({str(r.get("effort_used")) for r in S if r["model"] == m and r["condition"] == c}) for c in SWEEP_CONDITIONS}
        print(f"  {m:34s} {e}")

    with open(os.path.join(OUT, "sweep_summary.csv"), "w", newline="") as f:
        w = csv.writer(f); w.writerow(["model", "condition", "mean_reasoning_tokens", "mean_output_tokens", "mean_latency_s", "n"])
        for m in SWEEP_MODELS:
            for c in SWEEP_CONDITIONS:
                rows = [r for r in S if r["model"] == m and r["condition"] == c]
                w.writerow([m, c, mean([r.get("reasoning_tokens") or 0 for r in rows]),
                            mean([r.get("completion_tokens") or 0 for r in rows]),
                            round(sum(r.get("latency_s") or 0 for r in rows) / len(rows), 1) if rows else "", len(rows)])
    print(f"\nwrote {os.path.join(OUT, 'sweep_summary.csv')}")
    print("\nRead: bare vs empty_sysprompt isolates the presence of a system message.")
    print("      identity_true vs identity_false isolates whether the name has to be correct.")
    print("      If neutral_sysprompt matches identity_true, the effect is not about identity at all.")
print("\n" + ledger_summary())

## What these can and cannot show

Both experiments are N=3 on small model sets. They are designed to rule explanations out, not to establish an effect. A difference that survives all five sweep conditions on one model is still one model. Treat anything here as a lead for a larger run, not a result.